# Model Case Scenario — Investment Simulation

This notebook simulates an investor that trades gold using cached out-of-sample forecasts written by `Modelling Baseline.ipynb` to `Results/{STEM}.predictions.csv`. For each loaded model we compute an equity curve over the out-of-sample window and compare it against a buy-and-hold benchmark.

**Strategy specification**

- Starting capital: \$1000
- Long / flat (no shorts)
- Long if `pred > 0`; flat if `pred < 0`; **hold previous position** if `pred` falls inside the 45–55th percentile of the full-sample `y_true` distribution
- Position earns the **realized log-return over the decision interval** (computed from `gold_close`)
- No transaction costs
- Decision strides:
  - **12 bars** (= 60 min — matches the forecast horizon, fully non-overlapping)
  - **5 bars** (= 25 min — re-evaluate more frequently, holding period < forecast horizon)

**Metrics reported per (model × stride)**

- Terminal value & total return
- Annualised Sharpe (estimated empirically from the data span)
- Turnover (number of position flips, including the initial entry from cash)
- Buy-and-hold equivalent on the same decision grid

**Caveats** (see Section 7 for details)

- Full-sample percentile thresholds introduce a mild lookahead (defensible, common in the literature).
- Zero transaction costs is an upper bound — appendix sensitivity recommended.


## 0. Backtest configuration

Edit these constants before re-executing. The notebook loads the baseline pipeline (data, features, model factories, walk-forward runner) via `nbformat` **without** running the heavy grid (`CELL 8` of the baseline).

In [ ]:
# ── Backtest configuration ────────────────────────────────────────────────────
STARTING_CASH        = 1000.0
DEADBAND_PERCENTILES = (0.45, 0.55)        # percentiles of full-sample y_true
STRIDES_BARS         = [12, 5]             # 12 bars = 60 min (matches horizon); 5 bars = 25 min

# Which (dataset, scheme) combo and which models to include in the backtest.
# Only models with cached prediction sidecars that match this config will load.
DATASET_FOR_BACKTEST       = "poly+trad"     # one of DATASET_SPECS keys (set after loading baseline)
WINDOW_SCHEME_FOR_BACKTEST = "fixed300"      # one of WINDOW_SCHEMES keys
MODELS_FOR_BACKTEST        = ["linear", "rf", "lasso_cv", "pls_ols", "rf_pls"]
# Add "lstm", "lstm_pls" if TensorFlow is installed and you want their curves too.

BASELINE_NOTEBOOK_PATH = "Modelling Baseline.ipynb"   # relative to this notebook


## 1. Load the baseline pipeline (without running the heavy grid)

We `exec` every code cell of `Modelling Baseline.ipynb` **except** the grid runner (`CELL 8`) and its summary cells (`CELL 9`, `CELL 9.1`). After this cell we have the full data pipeline (`gold`, `X`, `y`, `DATASET_SPECS`), the model factories (`BASE_MODEL_BUILDERS`, `build_model_registry`), and helper utilities such as `artefact_paths`, `compute_metrics`, and `walk_forward` available in the local namespace.

This notebook does **not** call the heavy walk-forward. Section 2 loads cached prediction sidecars written by the baseline.

In [5]:
#check if nbformat is installed; if not, install it via pip and import it;
#otherwise, just import it. This is needed to read and execute the baseline notebook cells.
try:
    import nbformat
except ImportError:
    import subprocess
    import sys
    print("nbformat not found. Installing it via pip...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nbformat"])

import nbformat as _nbf
import re as _re
from pathlib import Path as _Path

_baseline_path = _Path(BASELINE_NOTEBOOK_PATH)
if not _baseline_path.exists():
    raise FileNotFoundError(
        f"Baseline notebook not found at {_baseline_path.resolve()}.\n"
        "Run this notebook from the same directory as 'Modelling Baseline.ipynb'."
    )

_nb_base = _nbf.read(_baseline_path, as_version=4)
# Match the "# %% ── CELL N : ..." marker for cells we want to skip (heavy grid + summaries).
_skip_re = _re.compile(r"# %% .{0,4} CELL (8|9|9\.1) :")

_executed, _skipped = [], []
for _cell in _nb_base.cells:
    if _cell.cell_type != "code":
        continue
    if _skip_re.search(_cell.source):
        _first_line = _cell.source.splitlines()[0][:80] if _cell.source.splitlines() else "<empty>"
        _skipped.append(_first_line)
        continue
    exec(compile(_cell.source, str(_baseline_path), "exec"), globals())
    _first_line = _cell.source.splitlines()[0][:80] if _cell.source.splitlines() else "<empty>"
    _executed.append(_first_line)

print(f"\nExecuted {len(_executed)} cell(s) from {_baseline_path.name}; "
      f"skipped {len(_skipped)} (grid + summary).")
print("Skipped:")
for _s in _skipped:
    print(f"  - {_s}")


Using Bloomberg workbook: Data\Indicators Data bloomberg.xlsx
Loaded PLS_N_COMPONENTS   : 3
Artefact path             : Feature selection log\pls_n_components_latest.json
Artefact generated_at_utc :  2026-05-14T15:32:50Z
Artefact grid_searched    :  [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20]
✅ Dependencies loaded. TF: True
✅ Traditional predictor helpers loaded.
FE_INPUT_MODE  : raw_panel
Latest panel   : polymarket_panel_filtered_2026-05-07.csv  (data date = 2026-05-07)
X_raw shape    : (7171, 690)   range: 2026-04-01 00:00:00 → 2026-05-07 22:00:00
Latest Bloomberg stationary panel : bloomberg_panel_stationary_2026-05-07.csv  (data date = 2026-05-07)
Latest Bloomberg metadata        : bloomberg_stationarity_metadata_2026-05-07.json
Bloomberg stationarity: 9 differenced (9 logret, 0 diff), 0 kept in levels  | train_fraction=1.0 | max_ffill_bars=None
openpyxl is already installed.
Bloomberg sheets loaded: ['GOLD USD SPOT PER OZ', 'CRUDE OIL (WTI) FUTURES PRI

## 2. Load cached walk-forward predictions

This section loads the out-of-sample prediction series from `Results/{STEM}.predictions.csv`, written by `Modelling Baseline.ipynb`. It never trains inside this notebook. Models with missing or stale sidecars are skipped with a non-blocking warning, and the backtest continues with whatever loads.

**Operational order**

1. Apply these notebook edits.
2. Run `Modelling Baseline.ipynb` end-to-end once. It now writes a `.predictions.csv` sidecar for every run and automatically recomputes older cached runs that are missing one.
3. Run `Model_Case_Scenario.ipynb` to load those sidecars and simulate only the available models.

In [ ]:
# ── Load cached walk-forward predictions (no training) ────────────────────────
import pandas as pd

# Validate config against what the baseline registered.
if DATASET_FOR_BACKTEST not in DATASET_SPECS:
    raise KeyError(
        f"DATASET_FOR_BACKTEST={DATASET_FOR_BACKTEST!r} not registered. "
        f"Available: {list(DATASET_SPECS.keys())}"
    )
if WINDOW_SCHEME_FOR_BACKTEST not in WINDOW_SCHEMES:
    raise KeyError(
        f"WINDOW_SCHEME_FOR_BACKTEST={WINDOW_SCHEME_FOR_BACKTEST!r} unknown. "
        f"Available: {list(WINDOW_SCHEMES.keys())}"
    )

_ds = DATASET_SPECS[DATASET_FOR_BACKTEST]
_n_features = _ds["X"].shape[1]
print(f"Backtest dataset : {DATASET_FOR_BACKTEST}  X={_ds['X'].shape}")
print(f"Window scheme    : {WINDOW_SCHEME_FOR_BACKTEST}")
print(f"Data SHA1        : {DATA_HASH[:12]}...   panel_date={panel_date}\n")

_registry = build_model_registry(DATASET_FOR_BACKTEST, _ds)

predictions_by_model = {}   # model_name -> {"y_true", "y_pred", "timestamps"}
skipped = {}                # model_name -> reason

for _mname in MODELS_FOR_BACKTEST:
    if _mname not in _registry:
        skipped[_mname] = "not registered for this dataset"
        print(f"WARNING  {_mname}: not registered for dataset "
              f"'{DATASET_FOR_BACKTEST}' -- skipped.")
        continue

    _model_path, _meta_path = artefact_paths(
        _mname, WINDOW_SCHEME_FOR_BACKTEST, panel_date, _n_features,
        dataset_tag=DATASET_FOR_BACKTEST,
    )
    _pred_path = _meta_path.with_name(_meta_path.stem + ".predictions.csv")

    if not _pred_path.exists():
        skipped[_mname] = f"no prediction sidecar at {_pred_path}"
        print(f"WARNING  {_mname}: prediction sidecar not found -- skipped.\n"
              f"         expected: {_pred_path}")
        continue

    # Staleness check against the metadata's data hash, when metadata is available.
    if _meta_path.exists():
        try:
            _meta = json.loads(_meta_path.read_text())
            if _meta.get("data_sha1") != DATA_HASH:
                skipped[_mname] = "stale predictions (data hash mismatch)"
                print(f"WARNING  {_mname}: cached predictions are STALE "
                      f"(built on different data) -- skipped.")
                continue
        except Exception as _e:
            print(f"WARNING  {_mname}: could not read metadata ({_e}); "
                  f"loading sidecar without staleness check.")
    else:
        print(f"WARNING  {_mname}: metadata JSON missing; "
              f"loading sidecar without staleness check.")

    _df = pd.read_csv(_pred_path, parse_dates=["timestamp"])
    predictions_by_model[_mname] = {
        "y_true":     _df["y_true"].to_numpy(),
        "y_pred":     _df["y_pred"].to_numpy(),
        "timestamps": list(_df["timestamp"]),
    }
    _m = compute_metrics(predictions_by_model[_mname]["y_true"],
                         predictions_by_model[_mname]["y_pred"])
    print(f"OK       {_mname}: loaded {len(_df):,} obs from {_pred_path.name}  "
          f"(rmse={_m['rmse']:.3e}  dir_acc={_m['dir_acc']:.3f})")

print(f"\nLoaded {len(predictions_by_model)} model(s): {list(predictions_by_model)}")
if skipped:
    print(f"Skipped {len(skipped)} model(s):")
    for _k, _v in skipped.items():
        print(f"  - {_k}: {_v}")

if not predictions_by_model:
    raise RuntimeError(
        "No cached predictions were loaded. Run 'Modelling Baseline.ipynb' "
        "end-to-end first so it writes Results/<STEM>.predictions.csv for the "
        "models you want to backtest, then re-run this notebook.  Expected "
        f"dataset='{DATASET_FOR_BACKTEST}', scheme='{WINDOW_SCHEME_FOR_BACKTEST}', "
        f"n_features={_n_features}, panel_date='{panel_date}'."
    )

Backtest dataset : poly+trad  X=(7134, 734)  y=(7134,)
Window scheme    : fixed300  (('fixed', 300))
⚠️  Requested models not registered for this dataset: ['linear', 'rf', 'lasso_cv']
Models to run    : ['pls_ols', 'rf_pls']

▶ Running walk-forward: pls_ols
  [pls_ols/fixed300] step 1/6834 (train=300, test=1)
  [pls_ols/fixed300] step 51/6834 (train=300, test=1)
  [pls_ols/fixed300] step 101/6834 (train=300, test=1)
  [pls_ols/fixed300] step 151/6834 (train=300, test=1)
  [pls_ols/fixed300] step 201/6834 (train=300, test=1)
  [pls_ols/fixed300] step 251/6834 (train=300, test=1)
  [pls_ols/fixed300] step 301/6834 (train=300, test=1)
  [pls_ols/fixed300] step 351/6834 (train=300, test=1)
  [pls_ols/fixed300] step 401/6834 (train=300, test=1)
  [pls_ols/fixed300] step 451/6834 (train=300, test=1)
  [pls_ols/fixed300] step 501/6834 (train=300, test=1)
  [pls_ols/fixed300] step 551/6834 (train=300, test=1)
  [pls_ols/fixed300] step 601/6834 (train=300, test=1)
  [pls_ols/fixed300] step 651/

KeyboardInterrupt: 

## 3. Backtest simulator

For a given stride **S** (in 5-min bars):

1. **Decision timestamps** = prediction timestamps sub-sampled at stride S (so successive holding periods do not overlap).
2. **Realized log-return** for stride k is `log(gold_close[t_{k+1}]) − log(gold_close[t_k])` where `t_{k+1} = t_k + S` bars.
3. **Signal** at decision k:
   - if `pred_k ∈ [P45, P55]` → hold previous position (deadband; default is *cash* before the first non-deadband signal)
   - elif `pred_k > 0` → long
   - else → flat
4. **Position** earns the realized log-return for the next stride.
5. **Equity** is `STARTING_CASH × exp(cumsum(position × realized_logret))`.

The deadband percentiles are computed from the **full-sample** `y_true` (constant across the backtest); see Section 7 for the lookahead caveat.

In [ ]:
import numpy as np
import pandas as pd


def _full_sample_deadband(y_true_all, p_low, p_high):
    return float(np.quantile(y_true_all, p_low)), float(np.quantile(y_true_all, p_high))


def _annualisation_factor(decision_times):
    """Decision samples per year, inferred from the actual sampling span."""
    if len(decision_times) < 2:
        return 1.0
    span_seconds = (decision_times[-1] - decision_times[0]).total_seconds()
    if span_seconds <= 0:
        return 1.0
    seconds_per_year = 365.25 * 24 * 3600
    return len(decision_times) * seconds_per_year / span_seconds


def simulate_strategy(
    timestamps,
    y_pred,
    gold_close,
    stride,
    deadband_lo,
    deadband_hi,
    starting_cash=1000.0,
):
    """Long/flat backtest at a fixed non-overlapping stride.

    Parameters
    ----------
    timestamps : sequence of pd.Timestamp aligned with y_pred
    y_pred     : 1-D array of forecasts (forecast horizon = 60 min by construction)
    gold_close : pd.Series of gold close prices indexed by datetime
    stride     : positive integer, decision interval in 5-min bars
    deadband_lo, deadband_hi : float thresholds; if deadband_lo <= pred <= deadband_hi -> hold
    starting_cash : initial equity in same units used downstream
    """
    timestamps = pd.DatetimeIndex(timestamps)
    y_pred = np.asarray(y_pred, dtype=float)
    n = len(timestamps)

    decision_idx = np.arange(0, n - stride, stride, dtype=int)
    if len(decision_idx) == 0:
        raise ValueError(f"Not enough timestamps ({n}) for stride={stride}.")
    decision_times = timestamps[decision_idx]
    next_times     = timestamps[decision_idx + stride]
    decision_preds = y_pred[decision_idx]

    closes      = gold_close.reindex(decision_times).to_numpy(dtype=float)
    closes_next = gold_close.reindex(next_times).to_numpy(dtype=float)

    # If any close is missing (rare — usually only at panel edges), drop those steps.
    _valid = ~(np.isnan(closes) | np.isnan(closes_next))
    if not _valid.all():
        decision_times = decision_times[_valid]
        next_times     = next_times[_valid]
        decision_preds = decision_preds[_valid]
        closes         = closes[_valid]
        closes_next    = closes_next[_valid]

    realized_logrets = np.log(closes_next) - np.log(closes)

    positions = np.zeros(len(decision_preds), dtype=int)
    prev = 0
    for i, pred in enumerate(decision_preds):
        if deadband_lo <= pred <= deadband_hi:
            positions[i] = prev          # hold previous
        elif pred > 0:
            positions[i] = 1             # long
        else:
            positions[i] = 0             # flat
        prev = positions[i]

    strategy_logrets = positions * realized_logrets
    equity_strategy  = starting_cash * np.exp(np.cumsum(strategy_logrets))
    equity_bh        = starting_cash * np.exp(np.cumsum(realized_logrets))

    # Trade count = number of position transitions, counting the initial entry from cash.
    n_trades = int((np.diff(np.concatenate(([0], positions))) != 0).sum())

    ann = _annualisation_factor(decision_times)
    sharpe_strategy = (
        float(strategy_logrets.mean() / strategy_logrets.std() * np.sqrt(ann))
        if strategy_logrets.std() > 0 else 0.0
    )
    sharpe_bh = (
        float(realized_logrets.mean() / realized_logrets.std() * np.sqrt(ann))
        if realized_logrets.std() > 0 else 0.0
    )

    pct_time_long = float(positions.mean())

    return {
        "decision_times":   decision_times,
        "positions":        positions,
        "realized_logrets": realized_logrets,
        "strategy_logrets": strategy_logrets,
        "equity_strategy":  equity_strategy,
        "equity_bh":        equity_bh,
        "terminal_value":   float(equity_strategy[-1]),
        "total_return":     float(equity_strategy[-1] / starting_cash - 1),
        "bh_terminal":      float(equity_bh[-1]),
        "bh_total_return":  float(equity_bh[-1] / starting_cash - 1),
        "n_trades":         n_trades,
        "ann_factor":       ann,
        "sharpe":           sharpe_strategy,
        "bh_sharpe":        sharpe_bh,
        "pct_time_long":    pct_time_long,
    }


print("✅ Simulator loaded.")


## 4. Run simulations: models × strides

For each (model, stride) we run `simulate_strategy` and collect the result in `backtest_results[(model, stride)]`. A summary DataFrame is built for the table in Section 5.

In [ ]:
# Use any model's y_true to compute the deadband (all models share the same OOS y_true).
_first_res = next(iter(predictions_by_model.values()))
y_true_all = _first_res["y_true"]
_deadband_lo, _deadband_hi = _full_sample_deadband(y_true_all, *DEADBAND_PERCENTILES)

print(f"Full-sample y_true: mean={y_true_all.mean():.3e}  std={y_true_all.std():.3e}  "
      f"min={y_true_all.min():.3e}  max={y_true_all.max():.3e}")
print(f"Deadband ({DEADBAND_PERCENTILES[0]:.0%}–{DEADBAND_PERCENTILES[1]:.0%}) : "
      f"[{_deadband_lo:.3e}, {_deadband_hi:.3e}]\n")

# `gold` is the DataFrame built in CELL 3 of the baseline; gold_close is the spot close.
_gold_close = gold["gold_close"].sort_index()
_gold_close = _gold_close[~_gold_close.index.duplicated(keep="last")]

backtest_results = {}
summary_rows = []
for stride in STRIDES_BARS:
    stride_min = stride * BAR_MINUTES
    for mname, res in predictions_by_model.items():
        sim = simulate_strategy(
            timestamps=res["timestamps"],
            y_pred=res["y_pred"],
            gold_close=_gold_close,
            stride=stride,
            deadband_lo=_deadband_lo,
            deadband_hi=_deadband_hi,
            starting_cash=STARTING_CASH,
        )
        backtest_results[(mname, stride)] = sim
        summary_rows.append({
            "model":           mname,
            "stride_bars":     stride,
            "stride_min":      stride_min,
            "n_decisions":     len(sim["decision_times"]),
            "n_trades":        sim["n_trades"],
            "pct_time_long":   sim["pct_time_long"],
            "terminal":        sim["terminal_value"],
            "total_return":    sim["total_return"],
            "sharpe":          sim["sharpe"],
            "bh_terminal":     sim["bh_terminal"],
            "bh_total_return": sim["bh_total_return"],
            "bh_sharpe":       sim["bh_sharpe"],
        })

summary_df = (
    pd.DataFrame(summary_rows)
      .sort_values(["stride_bars", "model"])
      .reset_index(drop=True)
)
print("✅ Backtest complete. Summary:\n")
with pd.option_context("display.float_format", "{:,.4f}".format, "display.width", 160):
    print(summary_df.to_string(index=False))


## 5. Summary table

Saved to `Results/backtest_summary_{dataset}_{scheme}.csv` for inclusion in the thesis appendix.

In [ ]:
_out_summary = (
    Path("Results") /
    f"backtest_summary_{DATASET_FOR_BACKTEST}_{WINDOW_SCHEME_FOR_BACKTEST}.csv"
)
_out_summary.parent.mkdir(parents=True, exist_ok=True)
summary_df.to_csv(_out_summary, index=False)
print(f"Saved summary to {_out_summary}")
summary_df


## 6. Faceted equity-curve plot

One subplot per stride; one line per model plus a dashed buy-and-hold benchmark. The figure is saved as PNG to `Results/`.

In [ ]:
import matplotlib.pyplot as plt

n_strides = len(STRIDES_BARS)
fig, axes = plt.subplots(n_strides, 1, figsize=(11, 4.0 * n_strides), sharex=False)
if n_strides == 1:
    axes = [axes]

for ax, stride in zip(axes, STRIDES_BARS):
    stride_min = stride * BAR_MINUTES
    bh_plotted = False
    for mname in predictions_by_model.keys():
        sim = backtest_results[(mname, stride)]
        ax.plot(sim["decision_times"], sim["equity_strategy"], label=mname, linewidth=1.2)
        if not bh_plotted:
            ax.plot(
                sim["decision_times"],
                sim["equity_bh"],
                label="buy & hold",
                linewidth=1.4,
                linestyle="--",
                color="black",
                alpha=0.7,
            )
            bh_plotted = True
    ax.axhline(STARTING_CASH, color="grey", linewidth=0.7, alpha=0.5)
    ax.set_title(
        f"Equity curve — stride {stride} bars ({stride_min} min)  |  "
        f"dataset={DATASET_FOR_BACKTEST}  scheme={WINDOW_SCHEME_FOR_BACKTEST}"
    )
    ax.set_ylabel("Equity ($)")
    ax.legend(loc="best", fontsize=8, ncol=2)
    ax.grid(alpha=0.3)

axes[-1].set_xlabel("Date")
fig.tight_layout()

_out_fig = (
    Path("Results") /
    f"backtest_equity_curves_{DATASET_FOR_BACKTEST}_{WINDOW_SCHEME_FOR_BACKTEST}.png"
)
fig.savefig(_out_fig, dpi=140, bbox_inches="tight")
print(f"Saved plot to {_out_fig}")
plt.show()


## 7. Notes & caveats

- **No transaction costs.** Strategy returns are gross. Even a flat 1 bp per position flip can flip the sign of a high-turnover strategy's Sharpe. Inspect `n_trades` in the summary table to gauge sensitivity; consider an appendix run with a small cost (e.g. multiply the equity update by `(1 − cost)` on each flip).
- **Full-sample deadband percentile.** The 45–55th percentile of `y_true` is computed on the entire OOS sample, which uses knowledge that would not have been available in real time. The bias is small in practice — it only re-classifies *which* trades are no-ops, not the sign of any executed trade — but a robust appendix run can recompute the percentile from a rolling training-window estimate.
- **Sharpe annualisation.** Estimated empirically as `samples_per_year = n_decisions × (year / data_span)` and Sharpe is `mean / std × √samples_per_year`. This is robust to weekend / overnight gaps and irregular sampling but assumes the OOS window is representative of a "year" of returns.
- **Stride 5 ≠ forecast horizon.** When stride = 5 bars (25 min) the holding period is shorter than the model's 60-min forecast horizon, so consecutive *signals* overlap (each prediction targets the next hour, but the position is liquidated after 25 min). This is intentional — it tests whether the directional information in the forecasts is still useful at shorter holding periods.
- **Cached prediction dependency.** This notebook loads OOS predictions from `Results/{STEM}.predictions.csv` sidecars written by `Modelling Baseline.ipynb`. Run the baseline end-to-end first; models with missing or stale sidecars are skipped with a warning.
